# Notebook 05: Support Vector Machine

## Overview
This notebook trains a support vector machine with a radial basis function kernel on the PCA reduced features. A support vector machine looks for the boundary that separates the classes with the widest margin, and the radial basis kernel lets it draw curved boundaries. It works on the 50 component PCA features because a kernel machine is slow on the full high dimensional set and on the full 17,500 training rows. To keep tuning fast, the search for the best settings runs on a random subset of the training data, and the final model is then refitted on the whole training set.

## Objectives
1. Tune the C and gamma settings of the radial basis kernel on a training subset.
2. Refit the best model on the full training set and evaluate it on the test data.
3. Plot the confusion matrix and ROC curves.
4. Plot the learning curve and a validation curve over C.
5. Measure the train versus test accuracy gap and save the model and metrics.

In [ ]:
# ----- Environment setup -----
!pip install -q scikit-image seaborn joblib

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ---- Clone the project repository so notebooks can import from src/ ----
REPO       = "lung-colon-cancer-histopathology-ml"
REPO_LOCAL = "/content/" + REPO
REPO_URL   = "https://github.com/hodyek/" + REPO + ".git"

if not os.path.exists(REPO_LOCAL):
    !git clone $REPO_URL $REPO_LOCAL

# Insert at position 0 so our src/ always takes priority over any Colab defaults.
if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

# Quick sanity-check that the import will work before running any cells.
import importlib.util
_spec = importlib.util.find_spec("src.dataset")
if _spec is None:
    raise ImportError(
        "src.dataset not found. Check that the clone succeeded and "
        "that src/__init__.py exists in the repository."
    )
print("src modules found at:", os.path.join(REPO_LOCAL, "src"))

# ---- Project folders on Google Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/" + REPO
DATA_DIR   = os.path.join(DRIVE_ROOT, "data", "lung_colon_image_set")
FIG_DIR    = os.path.join(DRIVE_ROOT, "figures")
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
FEAT_DIR   = os.path.join(DRIVE_ROOT, "features")
for d in (FIG_DIR, MODEL_DIR, FEAT_DIR):
    os.makedirs(d, exist_ok=True)

CLASSES = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
print("Setup complete. CLASSES:", CLASSES)


In [ ]:
# Load the PCA features and labels from Notebook 02.
import numpy as np
y_train = np.load(os.path.join(FEAT_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(FEAT_DIR, "y_test.npy"))
Xtr_p = np.load(os.path.join(FEAT_DIR, "X_train_pca.npy"))
Xte_p = np.load(os.path.join(FEAT_DIR, "X_test_pca.npy"))
print("PCA train shape:", Xtr_p.shape)

In [ ]:
# Tune C and gamma on a random training subset to keep the kernel search fast.
from sklearn.svm import SVC
from src.train import tune, fit_and_time
from src.evaluate import get_scores, evaluate_model, print_metrics

rng = np.random.RandomState(42)
idx = rng.choice(len(Xtr_p), size=min(6000, len(Xtr_p)), replace=False)
Xsub, ysub = Xtr_p[idx], y_train[idx]

svm_grid = {"C": [1, 10], "gamma": ["scale", 0.01]}
svm_tuned, svm_params, t_search = tune(SVC(kernel="rbf"), svm_grid, Xsub, ysub, cv=3)
print("Best parameters:", svm_params, f"  search time: {t_search:.1f}s")

# Refit the best settings on the full training set.
svm = SVC(kernel="rbf", **svm_params)
svm, t_fit = fit_and_time(svm, Xtr_p, y_train)
print(f"Full refit time: {t_fit:.1f}s")

svm_pred = svm.predict(Xte_p)
svm_scores = get_scores(svm, Xte_p)   # decision_function for the SVM
svm_metrics = evaluate_model(y_test, svm_pred, svm_scores, CLASSES)
print_metrics("Support Vector Machine", svm_metrics)

The support vector machine reaches accuracy close to the random forest, showing that a margin based method also separates these tissue classes well on the reduced features. Tuning on a subset and refitting on the full set keeps the search fast while still using all the data for the final model. The radial basis kernel lets the boundary curve around the classes, which helps with the overlapping lung cases. As before, the metrics rise together, so the model is balanced across classes.

In [ ]:
# Confusion matrix and ROC curves.
from src.evaluate import plot_confusion_matrix, plot_roc_curves
plot_confusion_matrix(svm_metrics["cm"], CLASSES, "Support Vector Machine confusion matrix",
    os.path.join(FIG_DIR, "05_svm_confusion_matrix.png"))
plot_roc_curves(y_test, svm_scores, CLASSES, "Support Vector Machine ROC curves",
    os.path.join(FIG_DIR, "05_svm_roc.png"))

The confusion matrix is again mostly diagonal, with the small number of mistakes falling between the two lung cancer classes. This is now a clear and repeated theme across every model in the project. The ROC curves stay high for all classes, and the lowest curve usually belongs to one of the lung cancer classes. The agreement between models points to a genuine biological similarity rather than a weakness of any one method.

In [ ]:
# Learning curve and validation curve over C, both on a subset for speed.
from src.evaluate import plot_learning_curve, plot_validation_curve
from sklearn.svm import SVC

plot_learning_curve(SVC(kernel="rbf", **svm_params),
    Xsub, ysub, "Support Vector Machine learning curve (subset)",
    os.path.join(FIG_DIR, "05_svm_learning_curve.png"), cv=3)

plot_validation_curve(SVC(kernel="rbf", gamma=svm_params["gamma"]),
    Xsub, ysub, "C", [0.1, 1, 10, 100],
    "Support Vector Machine validation curve over C",
    os.path.join(FIG_DIR, "05_svm_validation_curve.png"), cv=3, logx=True)

The learning curve shows accuracy improving as more samples are used, with training and cross-validation lines close together, which is a healthy sign. The validation curve over C shows accuracy rising as the model is allowed to fit the data more tightly, then flattening once C is large enough. Very large C values risk overfitting, but the flat top here means a moderate C already captures the structure. These curves were computed on a subset to keep the kernel training time reasonable.

In [ ]:
# Overfitting gap and save.
from src.evaluate import overfitting_gap
from src.train import save_model, save_metrics
tr_acc, te_acc, gap = overfitting_gap(svm, Xtr_p, y_train, Xte_p, y_test)
print(f"Train accuracy: {tr_acc:.4f}  Test accuracy: {te_acc:.4f}  Gap: {gap:.4f}")
save_model(svm, os.path.join(MODEL_DIR, "svm_rbf.joblib"))
save_metrics(svm_metrics, os.path.join(MODEL_DIR, "svm_metrics.json"),
             extra={"model": "Support Vector Machine", "best_params": svm_params,
                    "train_time_s": t_fit, "train_acc": tr_acc, "test_acc": te_acc, "gap": gap})
print("Saved support vector machine model and metrics.")

The train to test gap for the support vector machine is small, which shows the chosen C and gamma do not overfit. A small gap together with high test accuracy means the margin based boundary generalises to new images. This matches the steady behaviour seen in the learning and validation curves. The model and its metrics are saved for the final comparison.

## Summary
The support vector machine performs on par with the random forest and clearly beats the baseline, working on just 50 PCA components. Its errors again sit between lung_aca and lung_scc, reinforcing the main difficulty of the task. The learning and validation curves are stable and the train to test gap is small. Tuning on a subset and refitting on all data kept the kernel training time manageable. The next notebook trains a gradient boosting model with XGBoost.